In [ ]:
import pandas as pd
from mopac_batch_runner import prepare_batch, run_mopac_batch, export_sdf, retry_failed_batch

In [2]:
df = pd.read_excel('./SMILES_v2.xlsx')
df

,Ion ID,SMILES
0,Cat1,CN=C([NH+](C)C)N(C)C
1,Cat2,CCCC[N+]1(CCCC1)C
2,Cat3,CCCCN1C=C[N+](=C1)C
3,Cat4,CCCCCCCCCCN1C=C[N+](=C1)C
4,Cat5,CCCCCCCCCCCCN1C=C[N+](=C1)C
...,...,...
78,An52,C(=C(C#N)C#N)=[N-]
79,An53,CCCCCCCCCCCCCOP(=O)(O)[O-]
80,An54,C(=O)(C(F)(F)F)[O-]
81,An55,C(F)(F)(F)S(=O)(=O)[O-]


In [3]:
df_smiles = df[['Ion ID', 'SMILES']]

In [4]:
work_dir = 'ILsy'

In [5]:
# Step 1: SMILES → 3D structures → .mop files
prep = prepare_batch(
    df=df_smiles,
    smiles_col="SMILES",
    id_col="Ion ID",
    keywords="PM7 PRECISE GEO-OK",
    workdir=work_dir,
    auto_charge=True,   # ← detects [NH4+], [O-], etc. automatically
    overwrite=True,
    max_workers=6,
)

In [6]:
# verify if anything failed
print(prep[prep['status'] != 'ready'])

Empty DataFrame
Columns: [compound_id, smiles, status, mop_file, error_message, duration_s]
Index: []


In [12]:
# Step 2: run MOPAC on all prepared .mop files
results = run_mopac_batch(
    workdir=work_dir,
    overwrite=True,
    timeout=1900,
    max_workers=14,
)

In [9]:
results[results['arc_file'].isna()]#.to_excel('twice_failed_molecules.xlsx')
results[results['arc_file'].isna()]['compound_id']

37    An43
Name: compound_id, dtype: str

In [ ]:
# Step 2b: retry any compound that failed or timed out, with a longer timeout.
# No-op (returns `results` unchanged) if nothing needs retrying.
results = retry_failed_batch(
    df=df_smiles,
    smiles_col="SMILES",
    id_col="Ion ID",
    keywords="PM7 PRECISE GEO-OK",
    workdir=work_dir,
    auto_charge=True,
    mopac_timeout=3600,   # give the stragglers more room than the original 1900s
    max_workers=6,
)

In [ ]:
# verify anything still unresolved after retry
results[results['arc_file'].isna()][['compound_id', 'status', 'error_message']]

In [14]:
# Step 3: export optimised geometries to SDF
sdf = export_sdf(
    workdir=work_dir,
    overwrite=True,
    max_workers=6,
)

In [15]:
print(sdf[sdf['status'] != 'success'])

Empty DataFrame
Columns: [compound_id, arc_file, sdf_file, status, error_message, duration_s]
Index: []


In [16]:
from pathlib import Path
import shutil

source_dir = Path(r"C:\Users\Maciej\Documents\GitHub\smi_to_mop\ILsy")
target_dir = Path(r"C:\Users\Maciej\Documents\GitHub\smi_to_mop\sdf_all")

target_dir.mkdir(exist_ok=True)

for sdf_file in source_dir.rglob("*.sdf"):
    shutil.copy2(sdf_file, target_dir / sdf_file.name)